# SECE Demo Notebook

This notebook demonstrates the SECE (Spatial Entropy-based Contrast Enhancement) and SECEDCT algorithms.

In [ ]:
# Install if needed
# !pip install sece

import numpy as np
import cv2
import matplotlib.pyplot as plt
from sece import sece, secedct, sece_simple, secedct_simple
from sece import color_sece, color_secedct
from sece.metrics import emeg, ssim, gmsd
from sece.baselines import ghe, clahe, wthe

print("SECE Demo - Spatial Entropy-based Contrast Enhancement")

## 1. Load Test Image

We'll use a sample image from scikit-image.

In [ ]:
from skimage import data
from skimage.color import rgb2gray

# Load sample image
image_rgb = data.astronaut()
image = (rgb2gray(image_rgb) * 255).astype(np.uint8)

# Create a low-contrast version for demonstration
low_contrast = np.clip(image.astype(np.float64) * 0.3 + 80, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')
axes[1].imshow(low_contrast, cmap='gray')
axes[1].set_title('Low Contrast Version')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 2. SECE Global Enhancement

Apply SECE for global contrast enhancement.

In [ ]:
# Apply SECE to low-contrast image
sece_result = sece(low_contrast)

print(f"Processing time: {sece_result.processing_time_ms:.2f} ms")
print(f"Number of gray levels: {len(sece_result.gray_levels)}")
print(f"Output range: [{sece_result.image.min()}, {sece_result.image.max()}]")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(low_contrast, cmap='gray')
axes[0].set_title('Low Contrast Input')
axes[0].axis('off')
axes[1].imshow(sece_result.image, cmap='gray')
axes[1].set_title('SECE Enhanced')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3. SECEDCT Local Enhancement

Compare different gamma values for local contrast control.

In [ ]:
# Compare different gamma values
gammas = [0.0, 0.3, 0.5, 0.7, 1.0]
results = {}

for gamma in gammas:
    result = secedct(low_contrast, gamma=gamma)
    results[gamma] = result
    print(f"gamma={gamma}: alpha={result.alpha:.3f}, time={result.processing_time_ms:.2f}ms")

# Visualize
fig, axes = plt.subplots(1, len(gammas), figsize=(15, 3))
for i, gamma in enumerate(gammas):
    axes[i].imshow(results[gamma].image, cmap='gray')
    axes[i].set_title(f'gamma={gamma}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 4. Histogram Comparison

SECE preserves histogram shape unlike traditional histogram equalization.

In [ ]:
# Compare histograms
ghe_result = ghe(low_contrast)
clahe_result = clahe(low_contrast)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))

# Row 1: Images
images = [low_contrast, sece_result.image, ghe_result, clahe_result]
titles = ['Original', 'SECE', 'GHE', 'CLAHE']
for i, (img, title) in enumerate(zip(images, titles)):
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(title)
    axes[0, i].axis('off')

# Row 2: Histograms
for i, (img, title) in enumerate(zip(images, titles)):
    axes[1, i].hist(img.ravel(), bins=256, range=(0, 256), color='steelblue', alpha=0.7)
    axes[1, i].set_title(f'{title} Histogram')
    axes[1, i].set_xlim(0, 256)

plt.tight_layout()
plt.show()

## 5. Quality Metrics Comparison

Compare enhancement quality using EMEG (higher = better contrast).

In [ ]:
# Compute EMEG for all methods
methods = {
    'Original': low_contrast,
    'SECE': sece_result.image,
    'SECEDCT (0.5)': results[0.5].image,
    'SECEDCT (1.0)': results[1.0].image,
    'GHE': ghe_result,
    'CLAHE': clahe_result,
}

emeg_scores = {name: emeg(img) for name, img in methods.items()}

# Bar chart
plt.figure(figsize=(10, 5))
bars = plt.bar(emeg_scores.keys(), emeg_scores.values(), color='steelblue')
plt.ylabel('EMEG Score')
plt.title('Enhancement Quality Comparison (Higher = Better)')
plt.xticks(rotation=45)

# Add value labels
for bar, score in zip(bars, emeg_scores.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
             f'{score:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Color Image Enhancement

Enhance color images while preserving hue.

In [ ]:
# Color enhancement
from cv2 import cvtColor, COLOR_RGB2BGR, COLOR_BGR2RGB

# Create low-contrast color image
low_contrast_color = np.clip(image_rgb.astype(np.float64) * 0.3 + 80, 0, 255).astype(np.uint8)
low_contrast_bgr = cvtColor(low_contrast_color, COLOR_RGB2BGR)

# Enhance with different color spaces
hsv_result = color_secedct(low_contrast_bgr, gamma=0.5, color_space='hsv')
lab_result = color_secedct(low_contrast_bgr, gamma=0.5, color_space='lab')
ycbcr_result = color_secedct(low_contrast_bgr, gamma=0.5, color_space='ycbcr')

# Convert back to RGB for display
def to_rgb(bgr):
    return cvtColor(bgr, COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(low_contrast_color)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(to_rgb(hsv_result.image))
axes[1].set_title(f'HSV ({hsv_result.processing_time_ms:.1f}ms)')
axes[1].axis('off')
axes[2].imshow(to_rgb(lab_result.image))
axes[2].set_title(f'LAB ({lab_result.processing_time_ms:.1f}ms)')
axes[2].axis('off')
axes[3].imshow(to_rgb(ycbcr_result.image))
axes[3].set_title(f'YCbCr ({ycbcr_result.processing_time_ms:.1f}ms)')
axes[3].axis('off')
plt.tight_layout()
plt.show()

## 7. Structural Similarity Analysis

Check how much structure is preserved vs. original.

In [ ]:
# SSIM comparison (comparing enhanced to original low-contrast)
ssim_scores = {}
for name, img in methods.items():
    if name != 'Original':
        ssim_scores[name] = ssim(low_contrast, img)

# GMSD comparison
gmsd_scores = {}
for name, img in methods.items():
    if name != 'Original':
        gmsd_scores[name] = gmsd(low_contrast, img)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# SSIM
bars1 = axes[0].bar(ssim_scores.keys(), ssim_scores.values(), color='forestgreen')
axes[0].set_ylabel('SSIM')
axes[0].set_title('Structural Similarity (Higher = More Similar)')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=45)
for bar, score in zip(bars1, ssim_scores.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                 f'{score:.3f}', ha='center', va='bottom', fontsize=9)

# GMSD
bars2 = axes[1].bar(gmsd_scores.keys(), gmsd_scores.values(), color='coral')
axes[1].set_ylabel('GMSD')
axes[1].set_title('Gradient Distortion (Lower = Less Distortion)')
axes[1].tick_params(axis='x', rotation=45)
for bar, score in zip(bars2, gmsd_scores.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, 
                 f'{score:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Summary

### Key Findings:

1. **SECE** provides artifact-free global contrast enhancement
2. **SECEDCT** adds local contrast control via gamma parameter
3. **Histogram preservation**: SECE maintains histogram shape unlike GHE
4. **Color preservation**: HSV processing preserves hue during enhancement
5. **Quality metrics**: EMEG shows improved contrast, SSIM shows structure preservation